In [1]:
# ================================
# 1️⃣ Install dependencies
# ================================
!pip install -qU "google-genai>=1.0.0" chromadb pypdf

In [2]:
# ================================
# 2️⃣ Imports and Gemini setup
# ================================
from pypdf import PdfReader
import chromadb
from google import genai
from google.colab import userdata

In [3]:
# Initialize the Gemini client
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

In [4]:
# ================================
# 3️⃣ Load PDF from Colab directory
# ================================
pdf_path = "/content/sample_data/AI_in_modern_society.pdf"
reader = PdfReader(pdf_path)

raw_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text += text + "\n"

print("PDF loaded. Preview:")
print(raw_text[:1000])  # first 1000 chars

PDF loaded. Preview:
Artiﬁcial Intelligence in Modern Society 
 
Introduction to Artiﬁcial Intelligence 
Artiﬁcial Intelligence (AI) refers to the simulation of human intelligence in 
machines that are programmed to think, learn, and make decisions. AI 
systems can perform tasks such as speech recognition, problem-solving, 
and pattern detection. 
AI has evolved signiﬁcantly since its inception in the mid-20th century. 
Early AI systems were rule-based, relying on predeﬁned logic. Modern AI 
systems, however, use machine learning and deep learning techniques to 
improve performance over time. 
Machine learning, a subset of AI, enables systems to learn from data 
without explicit programming. Deep learning, a specialized branch of 
machine learning, uses neural networks with multiple layers to analyze 
complex patterns. 
Today, AI is widely used in industries such as healthcare, ﬁnance, 
transportation, and entertainment. 
 
Applications of AI 
AI has numerous real-world applications: 


In [5]:
# ================================
# 4️⃣ Chunk the document
# ================================
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print("Total chunks created:", len(chunks))
print("\n--- Content of Each Chunk ---")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n--------------------")
print("----------------------------")

Total chunks created: 8

--- Content of Each Chunk ---
Chunk 1:
Artiﬁcial Intelligence in Modern Society 
 
Introduction to Artiﬁcial Intelligence 
Artiﬁcial Intelligence (AI) refers to the simulation of human intelligence in 
machines that are programmed to think, learn, and make decisions. AI 
systems can perform tasks such as speech recognition, problem-solving, 
and pattern detection. 
AI has evolved signiﬁcantly since its inception in the mid-20th century. 
Early AI systems were rule-based, relying on predeﬁned logic. Modern AI 
systems, however, use ma
--------------------
Chunk 2:
. 
Early AI systems were rule-based, relying on predeﬁned logic. Modern AI 
systems, however, use machine learning and deep learning techniques to 
improve performance over time. 
Machine learning, a subset of AI, enables systems to learn from data 
without explicit programming. Deep learning, a specialized branch of 
machine learning, uses neural networks with multiple layers to analyze 
complex patte

In [6]:
# ================================
# 6️⃣ Store chunks in ChromaDB
# ================================
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="simple_rag")

collection.add(
    documents=chunks,
    ids=[f"id_{i}" for i in range(len(chunks))]
)

print("Documents added to ChromaDB.")

# /root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz added to chroma db


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 44.7MiB/s]


Documents added to ChromaDB.


In [7]:
import google.generativeai as genai

# --- 4. Simple RAG Query ---
while True:
    user_query = input("Ask a question (type 'exit' to quit): ")

    if user_query.lower() == 'exit':
        print("Exiting Q&A session.")
        break

    # Retrieve top 2 matches
    results = collection.query(query_texts=[user_query], n_results=2)
    context = " ".join(results['documents'][0])

    print(f"\nRetrieved chunks content:\n{results['documents'][0]}") # Added line to print chunks content

    # Generate Final Answer
    prompt = f"Context: {context}\n\nQuestion: {user_query}\n\nAnswer:"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt)

    # Correctly display the number of retrieved chunks (n_results)
    print(f"Retrieved {len(results['documents'][0])} chunks from the knowledge base.")
    print(f"\nAnswer: {response.text}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Ask a question (type 'exit' to quit): How does AI improve efficiency in businesses?

Retrieved chunks content:
['AI oƯers several advantages: \n\uf0b7 EƯiciency: Automates repetitive tasks, saving time and resources \n\uf0b7 Accuracy: Reduces human error in data processing \n\uf0b7 Scalability: Handles large datasets eƯectively \n\uf0b7 Innovation: Enables new products and services \nAI-powered automation allows businesses to operate more eƯiciently and \nfocus on strategic tasks. It also enhances decision-making by providing \ndata-driven insights. \n \nChallenges and Ethical Concerns \nDespite its beneﬁts, AI raises several concerns:', ' trading, and risk assessment. It \ncan process vast amounts of ﬁnancial data quickly. \nTransportation: \nSelf-driving cars use AI to navigate roads, detect obstacles, and make real-\ntime decisions. \nRetail: \nAI improves customer experience through recommendation systems and \nchatbots. \nEducation: \nAI enables personalized learning by adapting c